In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import pandas as pd
import numpy as np
import random
import os
import sys

from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    average_precision_score,
    f1_score,
    confusion_matrix
)

In [ ]:
IS_COLAB = "google.colab" in sys.modules

if IS_COLAB:
    print("Running on Google Colab")

    from google.colab import drive
    drive.mount('/content/drive')

    BASE_PATH = "/content/drive/MyDrive/DGvGAN"

else:
    print("Running on Local Machine")

    BASE_PATH = os.getcwd()

DATA_PATH = os.path.join(BASE_PATH,"dataset.csv")
CHECKPOINT_DIR = os.path.join(BASE_PATH,"checkpoints")
MODEL_DIR = os.path.join(BASE_PATH,"models")
LOG_FILE = os.path.join(BASE_PATH,"training_logs.csv")

os.makedirs(CHECKPOINT_DIR,exist_ok=True)
os.makedirs(MODEL_DIR,exist_ok=True)

print("Dataset:",DATA_PATH)

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
NUM_API_CALLS = 307
SEQ_LEN = 100

LATENT_DIM = 128
EMB_DIM = 128

BATCH_SIZE = 32
EPOCHS = 50

SEEDS = [10,20,30,40,50]

In [ ]:
def set_seed(seed):

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [ ]:
df = pd.read_csv(DATA_PATH)

train_df, test_df = train_test_split(
    df,
    test_size=0.3,
    stratify=df["malware"],
    random_state=42
)

In [ ]:
class MalwareGraphDataset(Dataset):

    def __init__(self, df):
        self.sequences = df.drop(columns=['hash','malware']).values
        self.labels = torch.tensor(df['malware'].values)

    def __getitem__(self, idx):
        seq = torch.tensor(self.sequences[idx])
        label = self.labels[idx]
        return seq, label

    def __len__(self):
        return len(self.labels)

In [ ]:
train_loader = DataLoader(
    MalwareGraphDataset(train_df),
    batch_size=BATCH_SIZE,
    shuffle=True
)

test_loader = DataLoader(
    MalwareGraphDataset(test_df),
    batch_size=BATCH_SIZE
)

In [ ]:
def seq_to_graph(seq_batch):

    B = seq_batch.size(0)

    src = seq_batch[:,:-1]
    dst = seq_batch[:,1:]

    adj = torch.zeros((B,NUM_API_CALLS,NUM_API_CALLS),device=seq_batch.device)

    batch_index = torch.arange(B,device=seq_batch.device).unsqueeze(1)

    adj[batch_index,src,dst]+=1

    X = F.one_hot(seq_batch,NUM_API_CALLS).float().permute(0,2,1)

    return adj,X

In [ ]:
class GATLayer(nn.Module):

    def __init__(self,in_features,out_features,heads=4):
        super().__init__()

        self.heads = heads
        self.out_features = out_features

        self.W = nn.Parameter(
            torch.randn(heads,in_features,out_features)*0.01
        )

        self.a_src = nn.Parameter(
            torch.randn(heads,out_features,1)*0.01
        )

        self.a_dst = nn.Parameter(
            torch.randn(heads,out_features,1)*0.01
        )

    def forward(self,adj,X):

        B,N,_ = adj.size()

        outputs = []

        for h in range(self.heads):

            Wh = X @ self.W[h]

            f1 = Wh @ self.a_src[h]
            f2 = Wh @ self.a_dst[h]

            e = f1 + f2.transpose(1,2)

            e = F.leaky_relu(e)

            zero_vec = -9e15*torch.ones_like(e)

            attention = torch.where(adj>0,e,zero_vec)

            attention = F.softmax(attention,dim=2)

            h_out = attention @ Wh

            outputs.append(h_out)

        H = torch.cat(outputs,dim=2)

        return H

In [ ]:
class GAT_Discriminator(nn.Module):

    def __init__(self):

        super().__init__()

        self.gat1 = GATLayer(SEQ_LEN,64,heads=4)
        self.gat2 = GATLayer(64*4,32,heads=4)

        self.dropout = nn.Dropout(0.5)

        self.fc = nn.Linear(NUM_API_CALLS*32*4,3)

    def forward(self,adj,X,return_features=False):

        Z = self.gat1(adj,X)
        Z = F.elu(Z)

        Z = self.gat2(adj,Z)
        Z = F.elu(Z)

        Z = self.dropout(Z)

        features = Z.reshape(Z.size(0),-1)

        logits = self.fc(features)

        if return_features:
            return logits,features

        return logits

In [ ]:
class Generator(nn.Module):

    def __init__(self):

        super().__init__()

        self.init_fc = nn.Linear(LATENT_DIM,256)

        self.rnn = nn.GRU(
            input_size=EMB_DIM,
            hidden_size=256,
            batch_first=True
        )

        self.token_proj = nn.Linear(256,NUM_API_CALLS)

        self.start_token = nn.Parameter(torch.zeros(1,1,EMB_DIM))

    def forward(self,z):

        B = z.size(0)

        h0 = torch.tanh(self.init_fc(z)).unsqueeze(0)

        inputs = self.start_token.repeat(B,SEQ_LEN,1)

        outputs,_ = self.rnn(inputs,h0)

        logits = self.token_proj(outputs)

        probs = F.gumbel_softmax(logits,tau=0.5,hard=True)

        tokens = torch.argmax(probs,dim=-1)

        return tokens

In [ ]:
class HybridModel(nn.Module):

    def __init__(self,G,D):

        super().__init__()

        self.G = G
        self.D = D

    def forward(self,seq):

        adj,X = seq_to_graph(seq)

        logits = self.D(adj,X)

        return logits

In [ ]:
results=[]
logs=[]
best_auc=0

for seed in SEEDS:

    print("\nRunning seed:",seed)

    set_seed(seed)

    G = Generator().to(DEVICE)
    D = GAT_Discriminator().to(DEVICE)

    model = HybridModel(G,D).to(DEVICE)

    opt_D = optim.Adam(D.parameters(),lr=2e-4)
    opt_G = optim.Adam(G.parameters(),lr=2e-4)

    for epoch in range(EPOCHS):

        model.train()

        for seq_real,labels in train_loader:

            seq_real = seq_real.to(DEVICE)
            labels = labels.to(DEVICE)

            adj_real,X_real = seq_to_graph(seq_real)

            B = labels.size(0)

            opt_D.zero_grad()

            logits_real = D(adj_real,X_real)
            loss_real = F.cross_entropy(logits_real,labels)

            z = torch.randn(B,LATENT_DIM).to(DEVICE)
            fake_seq = G(z)

            adj_fake,X_fake = seq_to_graph(fake_seq)

            logits_fake = D(adj_fake,X_fake)

            fake_labels = torch.full((B,),2,device=DEVICE)

            loss_fake = F.cross_entropy(logits_fake,fake_labels)

            loss_D = loss_real + loss_fake

            loss_D.backward()
            opt_D.step()

            opt_G.zero_grad()

            z = torch.randn(B,LATENT_DIM).to(DEVICE)
            fake_seq = G(z)

            adj_fake,X_fake = seq_to_graph(fake_seq)

            logits_fake,feat_fake = D(adj_fake,X_fake,True)
            _,feat_real = D(adj_real,X_real,True)

            loss_G = F.mse_loss(feat_fake.mean(0),feat_real.mean(0))

            loss_G.backward()
            opt_G.step()

        print(f"Epoch {epoch+1} | D {loss_D:.4f} | G {loss_G:.4f}")

        logs.append({
            "seed":seed,
            "epoch":epoch+1,
            "D_loss":loss_D.item(),
            "G_loss":loss_G.item()
        })

    model.eval()

    all_probs=[]
    all_labels=[]

    with torch.no_grad():

        for seq,labels in test_loader:

            seq = seq.to(DEVICE)

            logits = model(seq)

            probs = torch.softmax(logits[:,:2],dim=1)

            all_probs.extend(probs[:,1].cpu().numpy())
            all_labels.extend(labels.numpy())

    preds = [1 if x>0.5 else 0 for x in all_probs]

    macro_f1 = f1_score(all_labels,preds,average="macro")
    roc_auc = roc_auc_score(all_labels,all_probs)
    pr_auc = average_precision_score(all_labels,all_probs)

    torch.save({
        "seed":seed,
        "generator_state_dict":G.state_dict(),
        "discriminator_state_dict":D.state_dict(),
        "macro_f1":macro_f1,
        "roc_auc":roc_auc,
        "pr_auc":pr_auc
    }, os.path.join(CHECKPOINT_DIR,f"seed_{seed}.pt"))

    torch.save(model,os.path.join(MODEL_DIR,f"full_model_seed_{seed}.pt"))

    print(f"Saved model for seed {seed}")

    if roc_auc > best_auc:

        best_auc = roc_auc

        torch.save(model,os.path.join(MODEL_DIR,"best_full_model.pt"))

        print("New best model saved!")

In [ ]:
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    f1_score,
    accuracy_score,
    precision_score,
    recall_score,
    classification_report,
    confusion_matrix
)

print("Evaluation")

results = []

for seed in SEEDS:

    model_path = f"/content/drive/MyDrive/DGvGAN/models/full_model_seed_{seed}.pt"

    if not os.path.exists(model_path):
        print("Model not found:", model_path)
        continue


    print("Evaluating Seed:", seed)

    model = torch.load(model_path, map_location=DEVICE, weights_only=False)
    model.eval()

    all_probs = []
    all_labels = []

    with torch.no_grad():
        for seq, labels in test_loader:

            seq = seq.to(DEVICE)

            logits = model(seq)

            probs = torch.softmax(logits[:, :2], dim=1)

            all_probs.extend(probs[:,1].cpu().numpy())
            all_labels.extend(labels.numpy())

    preds = [1 if x > 0.5 else 0 for x in all_probs]

    # Metrics
    roc_auc = roc_auc_score(all_labels, all_probs)
    pr_auc = average_precision_score(all_labels, all_probs)
    acc = accuracy_score(all_labels, preds)

    macro_f1 = f1_score(all_labels, preds, average="macro")
    weighted_f1 = f1_score(all_labels, preds, average="weighted")

    precision_malware = precision_score(all_labels, preds, pos_label=1)
    recall_malware = recall_score(all_labels, preds, pos_label=1)

    precision_benign = precision_score(all_labels, preds, pos_label=0)
    recall_benign = recall_score(all_labels, preds, pos_label=0)

    cm = confusion_matrix(all_labels, preds)

    tn, fp, fn, tp = cm.ravel()

    false_positive_rate = fp / (fp + tn)

    print("\nOverall Metrics")
    print("Accuracy:", acc)
    print("ROC-AUC:", roc_auc)
    print("PR-AUC:", pr_auc)
    print("Macro F1:", macro_f1)
    print("Weighted F1:", weighted_f1)

    print("\nMalware Class Metrics")
    print("Precision (Malware):", precision_malware)
    print("Recall (Malware):", recall_malware)

    print("\nBenign Class Metrics")
    print("Precision (Benign):", precision_benign)
    print("Recall (Benign):", recall_benign)

    print("\nFalse Positive Rate:", false_positive_rate)

    print("\nConfusion Matrix")
    print(cm)

    print("\nClassification Report")
    print(classification_report(all_labels, preds, digits=4))

    results.append({
        "seed": seed,
        "accuracy": acc,
        "roc_auc": roc_auc,
        "pr_auc": pr_auc,
        "macro_f1": macro_f1,
        "fpr": false_positive_rate
    })


print("Summary Across Seeds")

results_df = pd.DataFrame(results)
print(results_df)

print("\nMean ROC-AUC:", results_df["roc_auc"].mean())
print("Std ROC-AUC :", results_df["roc_auc"].std())

In [ ]:
log_df = pd.DataFrame(logs)
log_df.to_csv(LOG_FILE,index=False)

print("Training logs saved:",LOG_FILE)